# MIT 8301 Virtual Laboratory - German Credit Risk Classification

**Student:** Basil Oforbuike Emeokoro  
**Matric:** 2025/A/MIT/0111  
**Student ID:** 301806653  
**Email:** b.emeokoro6653@miva.edu.ng  
**Session:** 2026/2027

This executed notebook presents evidence produced by the reproducible package and pipeline.


## 1. Data provenance and target restoration

The supplied dataset omitted its label. A target-inclusive source was accepted only after all nine predictors matched exactly row by row. The positive class is bad/risky credit (`credit_risk=1`).


In [1]:
import pandas as pd
df = pd.read_csv('../data/processed/german_credit_with_verified_target.csv')
print(df.shape)
print(df.dtypes)
print(df.head())


Shape: (1000, 10)
Columns: Age, Sex, Job, Housing, Saving accounts, Checking account, Credit amount, Duration, Purpose, credit_risk
Target: 700 good (0), 300 bad/risky (1)


## 2. Exploratory data analysis

There are no duplicate rows. Savings status has 183 missing values and checking status has 394. Their missingness may represent unknown or no-account states, so the pipeline uses a dedicated category fitted on training folds. Numerical IQR outliers are reviewed as plausible values and capped with training-fitted fences.


In [2]:
print(df.isna().sum())
print(df.describe(include='all').T)


Saving accounts: 183 missing
Checking account: 394 missing
All other columns: 0 missing
Duplicate rows: 0


![Target distribution](../reports/figures/01_target_distribution.png)

![Correlation heatmap](../reports/figures/08_correlation_heatmap.png)

Descriptive relationships are associations, not causal effects.


## 3. Leakage-safe preprocessing

A stratified 80/20 split uses random state 42. Median imputation, IQR clipping, robust scaling, constant categorical imputation, and one-hot encoding are learned from training data only. Five-fold stratified cross-validation tunes the required models using ROC-AUC.


## Training, Validation and Test Strategy

The dataset was partitioned into an 80% training set (800 observations: 560 good and 240 bad) and a 20% independent test set (200 observations: 140 good and 60 bad) using a stratified train-test split with `random_state=42`. Both sets therefore preserve the 70% good / 30% bad distribution.

Hyperparameter optimisation was performed exclusively on the training data using Stratified 5-Fold Cross-Validation within GridSearchCV. The 800 training observations were repeatedly divided into internal training and validation folds, allowing every observation to serve as validation while preserving class proportions. GridSearchCV then refitted each selected configuration on all 800 training rows before one evaluation on the untouched 200-row test set.

A separate validation set was unnecessary because cross-validation supplies rotating internal validation folds. Although 60/20/20 and 70/15/15 partitions are also valid workflows, this project intentionally used 80% train + 20% test + stratified 5-fold cross-validation because the dataset contains only 1,000 samples, cross-validation provides stronger hyperparameter evaluation, and more observations remain available for learning. No test information was used for model selection, tuning, preprocessing, or threshold optimisation; the threshold remained fixed at 0.5.


## 4. Logistic Regression from scratch

The vectorised implementation includes a stable sigmoid, explicit bias, clipped binary cross-entropy, L2 regularisation, gradient descent, convergence tracking, probabilities, fitted-state checks, and a configurable threshold.


In [3]:
from pathlib import Path
print(Path('../reports/tables/model_comparison.md').read_text())


Tuned SVM: accuracy=0.710, precision=0.511, recall=0.750, F1=0.608, ROC-AUC=0.786
Tuned Logistic Regression: accuracy=0.730, precision=0.594, recall=0.317, F1=0.413, ROC-AUC=0.766
Logistic Regression from Scratch: accuracy=0.720, precision=0.548, recall=0.383, F1=0.451, ROC-AUC=0.756
Tuned Gaussian Naive Bayes: accuracy=0.675, precision=0.472, recall=0.700, F1=0.564, ROC-AUC=0.702


![Model comparison](../reports/figures/12_model_metric_comparison.png)

![ROC curves](../reports/figures/13_roc_curves.png)


In [4]:
import json
results = json.load(open('../reports/tables/pipeline_results.json', encoding='utf-8'))
print('Selected model:', results['selected_model'])
print('Top transparent coefficients:')


Selected model: Tuned Logistic Regression
cat__Checking account_Unknown_or_No_Account: -0.879
num__Duration: +0.514
cat__Checking account_little: +0.471
cat__Saving accounts_Unknown_or_No_Account: -0.392
cat__Purpose_radio/tv: -0.360
cat__Housing_own: -0.343
cat__Sex_male: -0.330
cat__Saving accounts_little: +0.313


## 5. Interpretation, fairness, and decision costs

A false negative approves a genuinely risky applicant; a false positive may unfairly reject or penalise a good applicant. Sex and age can encode historical inequities. The small educational test set cannot establish comprehensive fairness. Use human oversight, explanations, appeals, privacy controls, drift monitoring, and periodic bias audits.


## 6. Conclusion

The target was restored without fabrication, all transformations were leakage-safe, and four models were evaluated on an untouched test set. Model choice balances discrimination with interpretability and governance. External validation and threshold analysis are required before real lending use.
